# 10 — Descarga de corridas → Drive

Resuelve los 19 accessions de `data/organismos.tsv` a corridas contra la ENA y
baja los `.sra` a `tesis/80_sra/`.

**Las sesiones de Colab se mueren, y eso es lo normal, no la excepción.** Este
notebook está hecho para eso: el estado del trabajo es qué archivos existen en
Drive, así que re-ejecutarlo retoma donde quedó. Usá `LIMITE` para que cada
sesión haga una tanda y termine.

Corré antes `00_setup.ipynb`.

## Preámbulo: montar Drive y clonar el repo

El repo es público, así que el clon no necesita credenciales. **Los notebooks
llaman a los scripts del repo en vez de reimplementarlos**: el criterio de
selección de corridas y el de verificación de ensamblados tienen que vivir en
un solo lugar, o dejan de ser reproducibles.

In [11]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/tesis')
CLON  = pathlib.Path('/content/tesis')
assert DRIVE.exists(), f'no veo {DRIVE} — ¿montaste la cuenta correcta?'
print('Drive OK:', DRIVE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive OK: /content/drive/MyDrive/tesis


In [12]:
import subprocess

REPO = 'youkonskernel-afk/tesis'
URL_ANON = 'https://github.com/' + REPO + '.git'

_AYUDA = (
    "No pude clonar de forma anonima y no hay GITHUB_TOKEN en los Secrets.",
    "Dos salidas, cualquiera sirve:",
    "  a) hacer el repo publico: Settings -> General -> Change visibility",
    "  b) crear un PAT de solo lectura y guardarlo como GITHUB_TOKEN en el",
    "     panel de Secrets de Colab (la llave a la izquierda), habilitando",
    "     el acceso para este notebook.",
)


def _sin_token(txt, secreto):
    # git incluye la URL en sus mensajes de error, y esa URL lleva el token.
    return txt.replace(secreto, '***') if secreto else txt


def clonar():
    if CLON.exists():
        r = subprocess.run(['git', '-C', str(CLON), 'pull', '--ff-only'],
                           capture_output=True, text=True)
        return 'ya estaba clonado; ' + (r.stdout.strip() or r.stderr.strip())

    # 1. Anonimo. Alcanza si el repo es publico.
    r = subprocess.run(['git', 'clone', '--depth', '1', URL_ANON, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode == 0:
        return 'clon anonimo (el repo es publico)'

    # 2. Con token de los Secrets de Colab. Para repo privado.
    tok = None
    try:
        from google.colab import userdata
        tok = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    if not tok:
        raise RuntimeError(chr(10).join(_AYUDA))

    url = 'https://x-access-token:' + tok + '@github.com/' + REPO + '.git'
    r = subprocess.run(['git', 'clone', '--depth', '1', url, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('el clon con token fallo: ' + _sin_token(r.stderr, tok))

    # Sin esto el token queda escrito en .git/config dentro de la VM.
    subprocess.run(['git', '-C', str(CLON), 'remote', 'set-url', 'origin', URL_ANON],
                   capture_output=True, text=True)
    return 'clon con token (el repo es privado)'


print(clonar())
print(subprocess.run(['git', '-C', str(CLON), 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout.strip())

ya estaba clonado; Updating f6c6096..a838735
Fast-forward
 .../notebooks/00_setup.ipynb                       | 368 ---------------------
 notebooks/10_descarga_runs.ipynb                   |  50 ++-
 scripts/fetch_runs.sh                              | 106 ++++--
 3 files changed, 121 insertions(+), 403 deletions(-)
 delete mode 100644 github-google-drive-setup-cwapri/notebooks/00_setup.ipynb
a838735 Encolar la descarga completa, con orden, corte por tiempo y progreso


In [13]:
import glob, os, subprocess, shutil

# CADA notebook de Colab corre en su propia VM: lo que instalo otro cuaderno no
# existe aca. Por eso esta celda esta en los cuatro y es idempotente: si las
# herramientas ya estan, no hace nada.
SRA_VER = '3.1.1'
URL = f'https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/{SRA_VER}/sratoolkit.{SRA_VER}-ubuntu64.tar.gz'


def sh(cmd, t=600):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=t)


def _en_path(ruta):
    if ruta and ruta not in os.environ['PATH']:
        os.environ['PATH'] = ruta + ':' + os.environ['PATH']


def instala_sra():
    # ya desempaquetado en esta VM de una corrida anterior de la celda
    c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')
    if c:
        _en_path(c[0]);
    if shutil.which('prefetch') and shutil.which('vdb-validate'):
        return 'ya estaba'

    r = sh(f'wget -q -O /tmp/sra.tar.gz "{URL}"')
    if r.returncode == 0 and sh('tar -xzf /tmp/sra.tar.gz -C /opt').returncode == 0:
        c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')
        if c:
            _en_path(c[0])
            return f'tarball oficial {SRA_VER}'

    # La version del tarball puede cambiar o desaparecer. apt es mas viejo, pero
    # aca solo se descarga y se valida: nada de esto entra en la tesis.
    if sh('apt-get -qq install -y sra-toolkit').returncode == 0 and shutil.which('prefetch'):
        return 'apt (version distinta del tarball)'

    raise RuntimeError(
        'No pude instalar sra-tools ni por tarball ni por apt. '
        'Revisa la version vigente en https://github.com/ncbi/sra-tools/wiki '
        'y ajusta SRA_VER.')


if not shutil.which('jq'):
    sh('apt-get -qq update'); sh('apt-get -qq install -y jq')
print('sra-tools:', instala_sra())

faltan = [b for b in ('prefetch', 'vdb-validate', 'jq', 'curl', 'git')
          if not shutil.which(b)]
if faltan:
    raise RuntimeError('faltan herramientas: ' + ', '.join(faltan))
print('herramientas OK:', 'prefetch vdb-validate jq curl git')

sra-tools: ya estaba
herramientas OK: prefetch vdb-validate jq curl git


In [14]:
SRA = DRIVE / '80_sra'; SRA.mkdir(parents=True, exist_ok=True)
STAGING = pathlib.Path('/content/sra_staging'); STAGING.mkdir(exist_ok=True)
MANIFIESTO_DRIVE = DRIVE / '00_manifiestos' / 'srr_manifest.tsv'
import shutil as _sh
print('destino :', SRA)
print('staging :', STAGING, f'({_sh.disk_usage("/content").free/1e9:.0f} GB libres)')

destino : /content/drive/MyDrive/tesis/80_sra
staging : /content/sra_staging (216 GB libres)


## 1. Manifiesto

Se genera con `scripts/fetch_runs.sh manifest`, que consulta la ENA y filtra a
datos de RNA: `library_source = TRANSCRIPTOMIC` —el filtro duro, porque varios
BioProjects mezclan corridas GENOMIC— y `SINGLE` para RNA-Seq, porque el PAIRED
de un proyecto de RNA-Seq no es sRNA-seq.

Se guarda una copia en Drive: el clon es efímero y el manifiesto define el
trabajo pendiente.

In [15]:
import subprocess, shutil as _sh
if MANIFIESTO_DRIVE.exists():
    print('ya hay manifiesto en Drive; lo reuso.')
    print('Para regenerarlo, borralo primero.')
    _sh.copy(MANIFIESTO_DRIVE, CLON / 'data' / 'srr_manifest.tsv')
else:
    r = subprocess.run(['./scripts/fetch_runs.sh', 'manifest'], cwd=CLON,
                       capture_output=True, text=True)
    print(r.stdout[-4000:]); print(r.stderr[-4000:])
    MANIFIESTO_DRIVE.parent.mkdir(parents=True, exist_ok=True)
    _sh.copy(CLON / 'data' / 'srr_manifest.tsv', MANIFIESTO_DRIVE)
    print('copia guardada en', MANIFIESTO_DRIVE)

ya hay manifiesto en Drive; lo reuso.
Para regenerarlo, borralo primero.


## 2. Encolar las descargas

Recorre **toda** la cola de pendientes. `prefetch` escribe primero en el disco
de la VM, se corre `vdb-validate`, y **recién ahí** se mueve a Drive: escribir
GB directo al FUSE de Drive es lento e inestable, y un `.sra` truncado no falla
ruidosamente —alinea de menos—, así que el que no valida se descarta y nunca
llega al destino.

`HORAS` hace que corte **solo**, antes de que Colab mate la sesión a mitad de
una descarga. Re-ejecutar la celda retoma donde quedó: el estado es qué
archivos existen en Drive, no un contador.

`ORDEN` decide qué se baja primero:

- `entrenamiento` — `gadmo`, `galga` y `maggi` antes que el resto. Son los
  únicos con positivos curados por MirGeneDB, o sea de los que depende que el
  modelo sirva. Es el default.
- `chico` — organismos con menos pendientes primero, para completar organismos
  enteros cuanto antes. Un organismo completo se puede alinear; uno a medias no.
- `alfabetico` — orden fijo, útil si querés que sea predecible.

In [ ]:
ORGANISMO = ''              # '' = todos, o 'prupe', 'gadmo', ...
ORDEN     = 'entrenamiento' # entrenamiento | chico | alfabetico
HORAS     = 3               # corta solo pasadas N horas; None = sin corte
LIMITE    = None            # tope de corridas; None = la cola entera

env = dict(os.environ,
           SRA_DEST=str(SRA),
           SRA_STAGING=str(STAGING),
           SRA_LEDGER=str(CLON / 'data' / 'sra_md5.tsv'),
           MANIFEST=str(CLON / 'data' / 'srr_manifest.tsv'))

cmd = ['./scripts/fetch_runs.sh', 'prefetch']
if ORGANISMO:
    cmd.append(ORGANISMO)
cmd += ['--orden', ORDEN]
if HORAS:
    cmd += ['--horas', str(HORAS)]
if LIMITE:
    cmd += ['-n', str(LIMITE)]

print(' '.join(cmd))
p = subprocess.Popen(cmd, cwd=CLON, env=env, text=True,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
for ln in p.stdout:
    print(ln, end='')
p.wait()

./scripts/fetch_runs.sh prefetch --orden entrenamiento --horas 3
cola: 411 corridas pendientes, orden=entrenamiento, corte a las 3h

== [1/411] gadmo SRR2039267 (23058925 reads)


## 3. Guardar el ledger

Los md5 van a git, no solo a Drive. Copiá esta salida a `data/sra_md5.tsv` en el
repo y commiteala.

In [ ]:
led = CLON / 'data' / 'sra_md5.tsv'
print(led.read_text() if led.exists() else '(vacío)')

## 4. Repetir

Volvé a correr la celda 2 hasta que `90_estado.ipynb` no muestre faltantes.
Cada tanda retoma sola, así que alcanza con re-ejecutarla.

**Colab Free no está pensado para trabajo desatendido largo**: el uso sostenido
lleva a throttling. Conviene espaciar las tandas en vez de encadenarlas.